In [2]:
# -*- coding: utf-8 -*-
"""
Optimasi Hyperparameter SVM Menggunakan Particle Swarm Optimization (PSO)
untuk Klasifikasi Sinyal Seismik (Gempa vs Noise) Mandiri
-------------------------------------------------------------------------
Skrip ini mengimplementasikan ekstraksi fitur berbasis statistik gelombang 
(STA/LTA, zero-crossing, RMS, kurtosis), klasifikasi menggunakan Support Vector 
Machine (SVM), dan pencarian parameter optimal (C dan gamma) menggunakan PSO.
"""

import numpy as np
import json
import logging
import os
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import random
from tqdm import tqdm

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK (DOMAIN-SPECIFIC FEATURE ENGINEERING)
# ==============================================================================
def extract_statistical_features(signal):
    """
    Mengekstrak fitur statistik standar seismologi dari sinyal mentah:
    - Root Mean Square (RMS) / Energi
    - Kurtosis & Skewness (Distribusi amplitudo transien)
    - Zero Crossing Rate (Frekuensi dominan)
    - Peak-to-Average Ratio
    """
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    
    # Zero crossing rate
    zcr = np.mean(np.diff(np.signbit(sig)))
    
    # Peak to Average Ratio
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. IMPLEMENTASI PARTICLE SWARM OPTIMIZATION (PSO) UNTUK SVM
# ==============================================================================
class Particle:
    def __init__(self, bounds):
        # Posisi: [log10(C), log10(gamma)] agar pencarian skala logaritmik stabil
        self.dim = len(bounds)
        self.position = np.array([random.uniform(b[0], b[1]) for b in bounds])
        self.velocity = np.array([random.uniform(-(b[1]-b[0])*0.1, (b[1]-b[0])*0.1) for b in bounds])
        self.best_position = np.copy(self.position)
        self.best_fitness = -float('inf')
        self.fitness = -float('inf')
        self.bounds = bounds

    def update_position(self):
        for i in range(self.dim):
            self.position[i] += self.velocity[i]
            self.position[i] = max(self.bounds[i][0], min(self.position[i], self.bounds[i][1]))

def pso_optimize_svm(X_train, y_train, X_val, y_val, num_particles=15, max_iter=20):
    """
    Mencari hyperparameter C dan gamma terbaik untuk SVM menggunakan PSO.
    """
    logger.info("Memulai optimasi SVM menggunakan PSO...")
    
    # Rentang parameter dalam skala log10: C [0.01, 100] -> [-2, 2], gamma [0.001, 10] -> [-3, 1]
    bounds = [(-2.0, 2.0), (-3.0, 1.0)]
    
    particles = [Particle(bounds) for _ in range(num_particles)]
    global_best_position = np.zeros(2)
    global_best_fitness = -float('inf')
    
    for iteration in range(max_iter):
        for particle in particles:
            C_val = 10 ** particle.position[0]
            gamma_val = 10 ** particle.position[1]
            
            # Evaluasi Fitness (Akurasi pada data validasi)
            try:
                svm = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
                svm.fit(X_train, y_train)
                preds = svm.predict(X_val)
                fitness = accuracy_score(y_val, preds)
            except Exception:
                fitness = 0.0
                
            particle.fitness = fitness
            
            if particle.fitness > particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = np.copy(particle.position)
                
            if particle.fitness > global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = np.copy(particle.position)
                
        # Update kecepatan dan posisi
        w, c1, c2 = 0.5, 1.5, 1.5
        for particle in particles:
            for i in range(len(bounds)):
                r1, r2 = random.random(), random.random()
                cognitive = c1 * r1 * (particle.best_position[i] - particle.position[i])
                social = c2 * r2 * (global_best_position[i] - particle.position[i])
                particle.velocity[i] = w * particle.velocity[i] + cognitive + social
            particle.update_position()
            
        logger.info(f"Iterasi PSO {iteration+1}/{max_iter} | Best Val Accuracy: {global_best_fitness*100:.2f}%")
        
    best_C = 10 ** global_best_position[0]
    best_gamma = 10 ** global_best_position[1]
    return best_C, best_gamma

# ==============================================================================
# 3. MAIN EXECUTION PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE KLASIFIKASI SEISMIK BERBASIS PSO-SVM ===")
    
    # Path dataset JSON Indonesia Bapak
    JSON_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json'
    NUM_POINTS = 700
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON tidak ditemukan di: {JSON_PATH}. Sesuaikan path di skrip.")
        exit()
        
    logger.info("Memuat dataset JSON...")
    with open(JSON_PATH, 'r') as f:
        raw_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(raw_data.items(), desc="Ekstraksi Fitur Statistik"):
        # Noise sample (Label = 0)
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        # Earthquake sample (Label = 1)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Normalisasi fitur (StandardScaler manual)
    mean = np.mean(X_features, axis=0)
    std = np.std(X_features, axis=0) + 1e-9
    X_features = (X_features - mean) / std
    
    # Split Dataset: Train (60%), Validation (20%), Test (20%)
    indices = np.arange(len(X_features))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features) * 0.6)
    n_val = int(len(X_features) * 0.2)
    
    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]
    
    X_train, y_train = X_features[train_idx], y_labels[train_idx]
    X_val, y_val = X_features[val_idx], y_labels[val_idx]
    X_test, y_test = X_features[test_idx], y_labels[test_idx]
    
    logger.info(f"Distribusi Data -> Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    
    # Jalankan PSO untuk mencari parameter SVM terbaik
    best_C, best_gamma = pso_optimize_svm(X_train, y_train, X_val, y_val, num_particles=10, max_iter=15)
    
    logger.info(f"Parameter Optimal Terpilih -> C: {best_C:.4f}, gamma: {best_gamma:.4f}")
    
    # Latih model akhir dengan parameter optimal pada gabungan (Train + Val)
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.hstack([y_train, y_val])
    
    final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
    final_svm.fit(X_train_full, y_train_full)
    
    # Evaluasi pada Test Set
    y_pred = final_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI AKHIR MODEL PSO-SVM")
    logger.info("="*50)
    logger.info(f" - Akurasi Pengujian Test Set : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

2026-07-22 13:17:05,683 - [INFO]: === PIPELINE KLASIFIKASI SEISMIK BERBASIS PSO-SVM ===
2026-07-22 13:17:05,684 - [INFO]: Memuat dataset JSON...
Ekstraksi Fitur Statistik: 100%|██████████| 5601/5601 [00:04<00:00, 1312.54it/s]
2026-07-22 13:17:11,861 - [INFO]: Distribusi Data -> Train: 6721, Val: 2240, Test: 2241
2026-07-22 13:17:11,862 - [INFO]: Memulai optimasi SVM menggunakan PSO...
2026-07-22 13:17:25,167 - [INFO]: Iterasi PSO 1/15 | Best Val Accuracy: 61.79%
2026-07-22 13:17:37,763 - [INFO]: Iterasi PSO 2/15 | Best Val Accuracy: 61.88%
2026-07-22 13:17:50,647 - [INFO]: Iterasi PSO 3/15 | Best Val Accuracy: 61.88%
2026-07-22 13:18:03,566 - [INFO]: Iterasi PSO 4/15 | Best Val Accuracy: 61.88%
2026-07-22 13:18:16,371 - [INFO]: Iterasi PSO 5/15 | Best Val Accuracy: 61.92%
2026-07-22 13:18:29,003 - [INFO]: Iterasi PSO 6/15 | Best Val Accuracy: 61.92%
2026-07-22 13:18:41,645 - [INFO]: Iterasi PSO 7/15 | Best Val Accuracy: 61.92%
2026-07-22 13:18:54,402 - [INFO]: Iterasi PSO 8/15 | Best V

In [3]:
# -*- coding: utf-8 -*-
"""
Optimasi Hyperparameter SVM Menggunakan Particle Swarm Optimization (PSO)
untuk Dataset STEAD 5000 1C (Path Kustom)
-------------------------------------------------------------------------
Skrip ini mengekstrak fitur statistik seismik, melatih Support Vector Machine (SVM),
dan mengoptimalkan parameter C serta gamma menggunakan algoritma PSO.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import random
from tqdm import tqdm

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    """
    Mengekstrak fitur statistik seismologi dari sinyal mentah STEAD:
    - Root Mean Square (RMS) / Energi
    - Kurtosis & Skewness
    - Zero Crossing Rate (ZCR)
    - Peak-to-Average Ratio (PAR)
    """
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. IMPLEMENTASI PSO UNTUK SVM
# ==============================================================================
class Particle:
    def __init__(self, bounds):
        self.dim = len(bounds)
        self.position = np.array([random.uniform(b[0], b[1]) for b in bounds])
        self.velocity = np.array([random.uniform(-(b[1]-b[0])*0.1, (b[1]-b[0])*0.1) for b in bounds])
        self.best_position = np.copy(self.position)
        self.best_fitness = -float('inf')
        self.fitness = -float('inf')
        self.bounds = bounds

    def update_position(self):
        for i in range(self.dim):
            self.position[i] += self.velocity[i]
            self.position[i] = max(self.bounds[i][0], min(self.position[i], self.bounds[i][1]))

def pso_optimize_svm(X_train, y_train, X_val, y_val, num_particles=15, max_iter=20):
    logger.info("Memulai optimasi SVM menggunakan PSO pada dataset STEAD...")
    bounds = [(-2.0, 2.0), (-3.0, 1.0)] # [log10(C), log10(gamma)]
    
    particles = [Particle(bounds) for _ in range(num_particles)]
    global_best_position = np.zeros(2)
    global_best_fitness = -float('inf')
    
    for iteration in range(max_iter):
        for particle in particles:
            C_val = 10 ** particle.position[0]
            gamma_val = 10 ** particle.position[1]
            
            try:
                svm = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
                svm.fit(X_train, y_train)
                preds = svm.predict(X_val)
                fitness = accuracy_score(y_val, preds)
            except Exception:
                fitness = 0.0
                
            particle.fitness = fitness
            if particle.fitness > particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = np.copy(particle.position)
                
            if particle.fitness > global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = np.copy(particle.position)
                
        w, c1, c2 = 0.5, 1.5, 1.5
        for particle in particles:
            for i in range(len(bounds)):
                r1, r2 = random.random(), random.random()
                cognitive = c1 * r1 * (particle.best_position[i] - particle.position[i])
                social = c2 * r2 * (global_best_position[i] - particle.position[i])
                particle.velocity[i] = w * particle.velocity[i] + cognitive + social
            particle.update_position()
            
        logger.info(f"Iterasi PSO STEAD {iteration+1}/{max_iter} | Best Val Accuracy: {global_best_fitness*100:.2f}%")
        
    return 10 ** global_best_position[0], 10 ** global_best_position[1]

# ==============================================================================
# 3. MAIN EXECUTION PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE PSO-SVM PADA DATASET STEAD 5000 1C ===")
    
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_20260719_062844.json'
    JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    NUM_POINTS = 700
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON STEAD tidak ditemukan di: {JSON_PATH}")
        exit()
        
    logger.info(f"Memuat file STEAD: {JSON_PATH}")
    with open(JSON_PATH, 'r') as f:
        stead_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(stead_data.items(), desc="Ekstraksi Fitur STEAD"):
        # Noise sample (Label = 0)
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        # Earthquake sample (Label = 1)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Normalisasi fitur
    mean = np.mean(X_features, axis=0)
    std = np.std(X_features, axis=0) + 1e-9
    X_features = (X_features - mean) / std
    
    # Split Dataset: Train (60%), Validation (20%), Test (20%)
    indices = np.arange(len(X_features))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features) * 0.6)
    n_val = int(len(X_features) * 0.2)
    
    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]
    
    X_train, y_train = X_features[train_idx], y_labels[train_idx]
    X_val, y_val = X_features[val_idx], y_labels[val_idx]
    X_test, y_test = X_features[test_idx], y_labels[test_idx]
    
    logger.info(f"Distribusi STEAD -> Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    
    # Optimasi PSO untuk STEAD
    best_C, best_gamma = pso_optimize_svm(X_train, y_train, X_val, y_val, num_particles=10, max_iter=15)
    logger.info(f"Parameter Optimal STEAD -> C: {best_C:.4f}, gamma: {best_gamma:.4f}")
    
    # Pelatihan Akhir & Evaluasi
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.hstack([y_train, y_val])
    
    final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
    final_svm.fit(X_train_full, y_train_full)
    
    y_pred = final_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI AKHIR MODEL PSO-SVM (STEAD)")
    logger.info("="*50)
    logger.info(f" - Akurasi Pengujian Test Set : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

2026-07-22 14:32:00,022 - [INFO]: === PIPELINE PSO-SVM PADA DATASET STEAD 5000 1C ===
2026-07-22 14:32:00,029 - [INFO]: Memuat file STEAD: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
Ekstraksi Fitur STEAD: 100%|██████████| 5000/5000 [00:03<00:00, 1375.28it/s]
2026-07-22 14:32:05,285 - [INFO]: Distribusi STEAD -> Train: 6000, Val: 2000, Test: 2000
2026-07-22 14:32:05,286 - [INFO]: Memulai optimasi SVM menggunakan PSO pada dataset STEAD...
2026-07-22 14:32:14,121 - [INFO]: Iterasi PSO STEAD 1/15 | Best Val Accuracy: 79.20%
2026-07-22 14:32:23,333 - [INFO]: Iterasi PSO STEAD 2/15 | Best Val Accuracy: 79.75%
2026-07-22 14:32:34,556 - [INFO]: Iterasi PSO STEAD 3/15 | Best Val Accuracy: 79.90%
2026-07-22 14:32:44,704 - [INFO]: Iterasi PSO STEAD 4/15 | Best Val Accuracy: 80.10%
2026-07-22 14:32:53,711 - [INFO]: Iterasi PSO STEAD 5/15 | Best Val Accuracy: 80.20%
2026-07-22 14:33:03,294 - [INFO]: Iterasi PSO STEAD 6/15 | Best Val Accuracy: 8

In [4]:
# -*- coding: utf-8 -*-
"""
Optimasi Hyperparameter SVM Menggunakan Particle Swarm Optimization (PSO)
untuk Dataset STEAD 5000 1C + Visualisasi Matriks Konfusi
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. IMPLEMENTASI PSO UNTUK SVM
# ==============================================================================
class Particle:
    def __init__(self, bounds):
        self.dim = len(bounds)
        self.position = np.array([random.uniform(b[0], b[1]) for b in bounds])
        self.velocity = np.array([random.uniform(-(b[1]-b[0])*0.1, (b[1]-b[0])*0.1) for b in bounds])
        self.best_position = np.copy(self.position)
        self.best_fitness = -float('inf')
        self.fitness = -float('inf')
        self.bounds = bounds

    def update_position(self):
        for i in range(self.dim):
            self.position[i] += self.velocity[i]
            self.position[i] = max(self.bounds[i][0], min(self.position[i], self.bounds[i][1]))

def pso_optimize_svm(X_train, y_train, X_val, y_val, num_particles=10, max_iter=15):
    logger.info("Memulai optimasi SVM menggunakan PSO pada dataset STEAD...")
    bounds = [(-2.0, 2.0), (-3.0, 1.0)] # [log10(C), log10(gamma)]
    
    particles = [Particle(bounds) for _ in range(num_particles)]
    global_best_position = np.zeros(2)
    global_best_fitness = -float('inf')
    
    for iteration in range(max_iter):
        for particle in particles:
            C_val = 10 ** particle.position[0]
            gamma_val = 10 ** particle.position[1]
            
            try:
                svm = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
                svm.fit(X_train, y_train)
                preds = svm.predict(X_val)
                fitness = accuracy_score(y_val, preds)
            except Exception:
                fitness = 0.0
                
            particle.fitness = fitness
            if particle.fitness > particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = np.copy(particle.position)
                
            if particle.fitness > global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = np.copy(particle.position)
                
        w, c1, c2 = 0.5, 1.5, 1.5
        for particle in particles:
            for i in range(len(bounds)):
                r1, r2 = random.random(), random.random()
                cognitive = c1 * r1 * (particle.best_position[i] - particle.position[i])
                social = c2 * r2 * (global_best_position[i] - particle.position[i])
                particle.velocity[i] = w * particle.velocity[i] + cognitive + social
            particle.update_position()
            
        logger.info(f"Iterasi PSO STEAD {iteration+1}/{max_iter} | Best Val Accuracy: {global_best_fitness*100:.2f}%")
        
    return 10 ** global_best_position[0], 10 ** global_best_position[1]

# ==============================================================================
# 3. MAIN EXECUTION & VISUALIZATION PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE PSO-SVM + VISUALISASI (STEAD 5000 1C) ===")
    
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_20260719_062844.json'
    JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    NUM_POINTS = 700
    
    SAVE_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/output_pso_svm'
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON STEAD tidak ditemukan di: {JSON_PATH}")
        exit()
        
    logger.info(f"Memuat file STEAD: {JSON_PATH}")
    with open(JSON_PATH, 'r') as f:
        stead_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(stead_data.items(), desc="Ekstraksi Fitur STEAD"):
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Normalisasi fitur
    mean = np.mean(X_features, axis=0)
    std = np.std(X_features, axis=0) + 1e-9
    X_features = (X_features - mean) / std
    
    # Split Dataset: Train (60%), Validation (20%), Test (20%)
    indices = np.arange(len(X_features))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features) * 0.6)
    n_val = int(len(X_features) * 0.2)
    
    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]
    
    X_train, y_train = X_features[train_idx], y_labels[train_idx]
    X_val, y_val = X_features[val_idx], y_labels[val_idx]
    X_test, y_test = X_features[test_idx], y_labels[test_idx]
    
    # Optimasi PSO untuk STEAD
    best_C, best_gamma = pso_optimize_svm(X_train, y_train, X_val, y_val, num_particles=10, max_iter=15)
    logger.info(f"Parameter Optimal STEAD -> C: {best_C:.4f}, gamma: {best_gamma:.4f}")
    
    # Pelatihan Akhir & Evaluasi
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.hstack([y_train, y_val])
    
    final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
    final_svm.fit(X_train_full, y_train_full)
    
    y_pred = final_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI AKHIR MODEL PSO-SVM (STEAD)")
    logger.info("="*50)
    logger.info(f" - Akurasi Pengujian Test Set : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # ==============================================================================
    # 4. VISUALISASI MATRIKS KONFUSI (PUBLICATION-READY)
    # ==============================================================================
    logger.info("Membuat visualisasi matriks konfusi...")
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'PSO-SVM Performance on STEAD (1C)\nAccuracy: {acc*100:.2f}% | C={best_C:.2f}, gamma={best_gamma:.2f}', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual', fontweight='bold', fontsize=12)
    
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    
    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, "confusion_matrix_pso_svm_stead.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    
    logger.info(f"Grafik visualisasi matriks konfusi berhasil disimpan di: {plot_path}")

2026-07-22 14:36:51,296 - [INFO]: === PIPELINE PSO-SVM + VISUALISASI (STEAD 5000 1C) ===
2026-07-22 14:36:51,311 - [INFO]: Memuat file STEAD: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
Ekstraksi Fitur STEAD: 100%|██████████| 5000/5000 [00:03<00:00, 1382.47it/s]
2026-07-22 14:36:56,608 - [INFO]: Memulai optimasi SVM menggunakan PSO pada dataset STEAD...
2026-07-22 14:37:05,941 - [INFO]: Iterasi PSO STEAD 1/15 | Best Val Accuracy: 75.90%
2026-07-22 14:37:14,462 - [INFO]: Iterasi PSO STEAD 2/15 | Best Val Accuracy: 76.05%
2026-07-22 14:37:22,793 - [INFO]: Iterasi PSO STEAD 3/15 | Best Val Accuracy: 79.20%
2026-07-22 14:37:32,221 - [INFO]: Iterasi PSO STEAD 4/15 | Best Val Accuracy: 80.15%
2026-07-22 14:37:44,078 - [INFO]: Iterasi PSO STEAD 5/15 | Best Val Accuracy: 80.15%
2026-07-22 14:37:54,133 - [INFO]: Iterasi PSO STEAD 6/15 | Best Val Accuracy: 80.15%
2026-07-22 14:38:03,961 - [INFO]: Iterasi PSO STEAD 7/15 | Best Val Accuracy: 80

In [5]:
# -*- coding: utf-8 -*-
"""
Optimasi Hyperparameter SVM Menggunakan Algoritma Metaheuristik EMCO 
(Elephant Migration Colony Optimization) untuk Dataset STEAD 5000 1C
-------------------------------------------------------------------------
Algoritma EMCO meniru perilaku migrasi gajah antar habitat menuju pemimpin koloni,
memanfaatkan memori historis, dan adaptasi lingkungan.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. IMPLEMENTASI ALGORITMA EMCO (ELEPHANT MIGRATION COLONY OPTIMIZATION)
# ==============================================================================
class ElephantColonyOptimizer:
    def __init__(self, n_agents, dim, bounds, alpha=0.5, beta=0.3, gamma_param=0.2):
        self.n_agents = n_agents
        self.dim = dim
        self.bounds = bounds
        self.alpha = alpha  # Pengaruh arah migrasi menuju pemimpin
        self.beta = beta    # Pengaruh memori historis migrasi
        self.gamma = gamma_param # Pengaruh adaptasi lingkungan
        
        # Inisialisasi populasi agen gajah: RANDOMINIT
        self.agents = np.array([
            [random.uniform(bounds[j][0], bounds[j][1]) for j in range(dim)]
            for _ in range(n_agents)
        ])
        # Inisialisasi memori migrasi: memoryPool
        self.memoryPool = np.copy(self.agents)
        self.history_fitness = []

    def evaluate_fitness(self, X_train, y_train, X_val, y_val):
        fitness_list = []
        for agent in self.agents:
            C_val = 10 ** agent[0]
            gamma_val = 10 ** agent[1]
            try:
                svm = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
                svm.fit(X_train, y_train)
                preds = svm.predict(X_val)
                fit = accuracy_score(y_val, preds)
            except Exception:
                fit = 0.0
            fitness_list.append(fit)
        return np.array(fitness_list)

    def optimize(self, X_train, y_train, X_val, y_val, max_iter=20):
        logger.info("Memulai optimasi SVM menggunakan EMCO (Elephant Migration Colony Optimization)...")
        
        best_overall_solution = None
        best_overall_fitness = -float('inf')

        for t in range(max_iter):
            # EVALUATEFITNESS
            fitness = self.evaluate_fitness(X_train, y_train, X_val, y_val)
            
            # SELECT LEADER (Agen dengan fitness tertinggi)
            leader_idx = np.argmax(fitness)
            leader = self.agents[leader_idx]
            
            if fitness[leader_idx] > best_overall_fitness:
                best_overall_fitness = fitness[leader_idx]
                best_overall_solution = np.copy(leader)

            # MIGRATE AGENTS & UPDATEMEMORY
            new_agents = []
            for i in range(self.n_agents):
                # Pilih memori historis acak dari pool
                mem_idx = random.randint(0, self.n_agents - 1)
                historical_memory = self.memoryPool[mem_idx]
                
                # Adaptasi lingkungan (stokastik lokal)
                environmental_adaptation = np.array([
                    random.uniform(-0.1, 0.1) * (self.bounds[j][1] - self.bounds[j][0])
                    for j in range(self.dim)
                ])
                
                # Persamaan Migrasi EMCO
                new_pos = (self.alpha * leader) + (self.beta * historical_memory) + (self.gamma * environmental_adaptation)
                
                # Batasi agar tetap dalam bounds
                for j in range(self.dim):
                    new_pos[j] = max(self.bounds[j][0], min(new_pos[j], self.bounds[j][1]))
                
                new_agents.append(new_pos)
                
            self.agents = np.array(new_agents)
            
            # UPDATEMEMORY: Perbarui pool memori jika agen baru lebih baik
            new_fitness = self.evaluate_fitness(X_train, y_train, X_val, y_val)
            for i in range(self.n_agents):
                if new_fitness[i] > fitness[i]:
                    self.memoryPool[i] = np.copy(self.agents[i])

            # historyFitness
            current_best_fit = np.max(fitness)
            self.history_fitness.append(float(current_best_fit))
            
            logger.info(f"Iterasi EMCO {t+1}/{max_iter} | Best Val Accuracy: {current_best_fit*100:.2f}%")

        return best_overall_solution, self.history_fitness

# ==============================================================================
# 3. MAIN EXECUTION & VISUALIZATION PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE EMCO-SVM + VISUALISASI (STEAD 5000 1C) ===")
    
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_20260719_062844.json'
    JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    NUM_POINTS = 700
    
    SAVE_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/output_emco_svm'
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON STEAD tidak ditemukan di: {JSON_PATH}")
        exit()
        
    logger.info(f"Memuat file STEAD: {JSON_PATH}")
    with open(JSON_PATH, 'r') as f:
        stead_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(stead_data.items(), desc="Ekstraksi Fitur STEAD"):
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Normalisasi fitur
    mean = np.mean(X_features, axis=0)
    std = np.std(X_features, axis=0) + 1e-9
    X_features = (X_features - mean) / std
    
    # Split Dataset: Train (60%), Validation (20%), Test (20%)
    indices = np.arange(len(X_features))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features) * 0.6)
    n_val = int(len(X_features) * 0.2)
    
    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]
    
    X_train, y_train = X_features[train_idx], y_labels[train_idx]
    X_val, y_val = X_features[val_idx], y_labels[val_idx]
    X_test, y_test = X_features[test_idx], y_labels[test_idx]
    
    # Jalankan EMCO untuk optimasi SVM (Dim = 2 untuk [log10(C), log10(gamma)])
    bounds = [(-2.0, 2.0), (-3.0, 1.0)]
    emco = ElephantColonyOptimizer(n_agents=10, dim=2, bounds=bounds, alpha=0.5, beta=0.3, gamma_param=0.2)
    best_pos, history_fit = emco.optimize(X_train, y_train, X_val, y_val, max_iter=15)
    
    best_C = 10 ** best_pos[0]
    best_gamma = 10 ** best_pos[1]
    logger.info(f"Parameter Optimal EMCO -> C: {best_C:.4f}, gamma: {best_gamma:.4f}")
    
    # Pelatihan Akhir & Evaluasi
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.hstack([y_train, y_val])
    
    final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
    final_svm.fit(X_train_full, y_train_full)
    
    y_pred = final_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI AKHIR MODEL EMCO-SVM (STEAD)")
    logger.info("="*50)
    logger.info(f" - Akurasi Pengujian Test Set : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # ==============================================================================
    # 4. VISUALISASI MATRIKS KONFUSI (PUBLICATION-READY)
    # ==============================================================================
    logger.info("Membuat visualisasi matriks konfusi...")
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'EMCO-SVM Performance on STEAD (1C)\nAccuracy: {acc*100:.2f}% | C={best_C:.2f}, gamma={best_gamma:.2f}', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual', fontweight='bold', fontsize=12)
    
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    
    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, "confusion_matrix_emco_svm_stead.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    
    logger.info(f"Grafik visualisasi matriks konfusi berhasil disimpan di: {plot_path}")

2026-07-22 14:48:48,207 - [INFO]: === PIPELINE EMCO-SVM + VISUALISASI (STEAD 5000 1C) ===
2026-07-22 14:48:48,222 - [INFO]: Memuat file STEAD: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
Ekstraksi Fitur STEAD: 100%|██████████| 5000/5000 [00:03<00:00, 1351.42it/s]
2026-07-22 14:48:53,669 - [INFO]: Memulai optimasi SVM menggunakan EMCO (Elephant Migration Colony Optimization)...
2026-07-22 14:49:10,591 - [INFO]: Iterasi EMCO 1/15 | Best Val Accuracy: 77.65%
2026-07-22 14:49:26,295 - [INFO]: Iterasi EMCO 2/15 | Best Val Accuracy: 77.45%
2026-07-22 14:49:41,776 - [INFO]: Iterasi EMCO 3/15 | Best Val Accuracy: 77.15%
2026-07-22 14:49:57,379 - [INFO]: Iterasi EMCO 4/15 | Best Val Accuracy: 77.05%
2026-07-22 14:50:12,992 - [INFO]: Iterasi EMCO 5/15 | Best Val Accuracy: 77.10%
2026-07-22 14:50:28,487 - [INFO]: Iterasi EMCO 6/15 | Best Val Accuracy: 76.65%
2026-07-22 14:50:43,913 - [INFO]: Iterasi EMCO 7/15 | Best Val Accuracy: 76.35%
2026-0

In [2]:
# -*- coding: utf-8 -*-
"""
Optimasi Hyperparameter SVM Tingkat Lanjut Menggunakan Particle Swarm Optimization (PSO)
untuk Dataset STEAD 5000 1C (Pencarian Ruang Parameter Lebih Luas)
-------------------------------------------------------------------------
Skrip ini memperluas ruang pencarian (bounds) C dan gamma, serta meningkatkan 
jumlah partikel dan iterasi PSO untuk mencapai akurasi maksimal.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. IMPLEMENTASI PSO LANJUTAN UNTUK SVM
# ==============================================================================
class AdvancedParticle:
    def __init__(self, bounds):
        self.dim = len(bounds)
        self.position = np.array([random.uniform(b[0], b[1]) for b in bounds])
        self.velocity = np.array([random.uniform(-(b[1]-b[0])*0.15, (b[1]-b[0])*0.15) for b in bounds])
        self.best_position = np.copy(self.position)
        self.best_fitness = -float('inf')
        self.fitness = -float('inf')
        self.bounds = bounds

    def update_position(self):
        for i in range(self.dim):
            self.position[i] += self.velocity[i]
            self.position[i] = max(self.bounds[i][0], min(self.position[i], self.bounds[i][1]))

def pso_optimize_svm_advanced(X_train, y_train, X_val, y_val, num_particles=20, max_iter=25):
    logger.info("Memulai optimasi lanjutan SVM menggunakan PSO (Ruang parameter diperluas)...")
    
    # Memperluas rentang log10: C [0.001, 1000] -> [-3.0, 3.0], gamma [0.0001, 100] -> [-4.0, 2.0]
    bounds = [(-3.0, 3.0), (-4.0, 2.0)]
    
    particles = [AdvancedParticle(bounds) for _ in range(num_particles)]
    global_best_position = np.zeros(2)
    global_best_fitness = -float('inf')
    
    # Adaptive Inertia Weight
    w_max, w_min = 0.9, 0.4
    
    for iteration in range(max_iter):
        w = w_max - (w_max - w_min) * (iteration / max_iter)
        
        for particle in particles:
            C_val = 10 ** particle.position[0]
            gamma_val = 10 ** particle.position[1]
            
            try:
                svm = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
                svm.fit(X_train, y_train)
                preds = svm.predict(X_val)
                fitness = accuracy_score(y_val, preds)
            except Exception:
                fitness = 0.0
                
            particle.fitness = fitness
            if particle.fitness > particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = np.copy(particle.position)
                
            if particle.fitness > global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = np.copy(particle.position)
                
        c1, c2 = 1.5, 1.5
        for particle in particles:
            for i in range(len(bounds)):
                r1, r2 = random.random(), random.random()
                cognitive = c1 * r1 * (particle.best_position[i] - particle.position[i])
                social = c2 * r2 * (global_best_position[i] - particle.position[i])
                particle.velocity[i] = w * particle.velocity[i] + cognitive + social
            particle.update_position()
            
        logger.info(f"Iterasi PSO Advanced {iteration+1}/{max_iter} | Best Val Accuracy: {global_best_fitness*100:.2f}%")
        
    return 10 ** global_best_position[0], 10 ** global_best_position[1]

# ==============================================================================
# 3. MAIN PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE PSO-SVM LANJUTAN (OPTIMASI HYPERPARAMETER) ===")
    
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_20260719_062844.json'
    JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    NUM_POINTS = 700
    
    SAVE_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/output_pso_svm_optimized'
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON STEAD tidak ditemukan di: {JSON_PATH}")
        exit()
        
    logger.info(f"Memuat file STEAD: {JSON_PATH}")
    with open(JSON_PATH, 'r') as f:
        stead_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(stead_data.items(), desc="Ekstraksi Fitur STEAD"):
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Normalisasi fitur
    mean = np.mean(X_features, axis=0)
    std = np.std(X_features, axis=0) + 1e-9
    X_features = (X_features - mean) / std
    
    # Split Dataset: Train (60%), Validation (20%), Test (20%)
    indices = np.arange(len(X_features))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features) * 0.6)
    n_val = int(len(X_features) * 0.2)
    
    X_train, y_train = X_features[indices[:n_train]], y_labels[indices[:n_train]]
    X_val, y_val = X_features[indices[n_train:n_train+n_val]], y_labels[indices[n_train:n_train+n_val]]
    X_test, y_test = X_features[indices[n_train+n_val:]], y_labels[indices[n_train+n_val:]]
    
    # Jalankan PSO Lanjutan
    best_C, best_gamma = pso_optimize_svm_advanced(X_train, y_train, X_val, y_val, num_particles=20, max_iter=25)
    logger.info(f"Parameter Optimal Baru -> C: {best_C:.4f}, gamma: {best_gamma:.4f}")
    
    # Pelatihan Akhir & Evaluasi
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.hstack([y_train, y_val])
    
    final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
    final_svm.fit(X_train_full, y_train_full)
    
    y_pred = final_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI MODEL PSO-SVM OPTIMIZED")
    logger.info("="*50)
    logger.info(f" - Akurasi Pengujian Test Set : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # Visualisasi
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'Optimized PSO-SVM on STEAD\nAccuracy: {acc*100:.2f}% | C={best_C:.2f}, gamma={best_gamma:.2f}', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual', fontweight='bold', fontsize=12)
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, "confusion_matrix_pso_svm_optimized.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    logger.info(f"Grafik tersimpan di: {plot_path}")

2026-07-22 15:05:00,455 - [INFO]: === PIPELINE PSO-SVM LANJUTAN (OPTIMASI HYPERPARAMETER) ===
2026-07-22 15:05:00,468 - [INFO]: Memuat file STEAD: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
Ekstraksi Fitur STEAD: 100%|██████████| 5000/5000 [00:03<00:00, 1340.95it/s]
2026-07-22 15:05:05,931 - [INFO]: Memulai optimasi lanjutan SVM menggunakan PSO (Ruang parameter diperluas)...
2026-07-22 15:05:25,231 - [INFO]: Iterasi PSO Advanced 1/25 | Best Val Accuracy: 78.10%
2026-07-22 15:05:44,503 - [INFO]: Iterasi PSO Advanced 2/25 | Best Val Accuracy: 80.25%
2026-07-22 15:06:19,427 - [INFO]: Iterasi PSO Advanced 3/25 | Best Val Accuracy: 80.75%
2026-07-22 15:07:09,359 - [INFO]: Iterasi PSO Advanced 4/25 | Best Val Accuracy: 80.75%
2026-07-22 15:07:49,512 - [INFO]: Iterasi PSO Advanced 5/25 | Best Val Accuracy: 80.75%
2026-07-22 15:08:20,650 - [INFO]: Iterasi PSO Advanced 6/25 | Best Val Accuracy: 80.75%
2026-07-22 15:09:04,438 - [INFO]: Itera

In [3]:
# -*- coding: utf-8 -*-
"""
Training dan Serialisasi (Saving) Model Optimal PSO-SVM untuk Inferensi
-------------------------------------------------------------------------
Skrip ini melatih model SVM dengan hyperparameter optimal pada keseluruhan 
data latih/uji, lalu menyimpannya ke dalam file menggunakan joblib agar 
bisa digunakan kembali untuk inferensi di perangkat target.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import random
from tqdm import tqdm
import joblib

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. MAIN TRAINING & SAVING PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PELATIHAN DAN PENYIMPANAN MODEL FINAL PSO-SVM ===")
    
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_20260719_062844.json'
    JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    NUM_POINTS = 700
    
    MODEL_SAVE_DIR = os.path.join(KEY_DATA_DIR, "saved_model_pso_svm")
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON STEAD tidak ditemukan di: {JSON_PATH}")
        exit()
        
    logger.info(f"Memuat file STEAD: {JSON_PATH}")
    with open(JSON_PATH, 'r') as f:
        stead_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(stead_data.items(), desc="Ekstraksi Fitur"):
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Hitung parameter normalisasi (mean & std) untuk disimpan bersama model
    mean_val = np.mean(X_features, axis=0)
    std_val = np.std(X_features, axis=0) + 1e-9
    X_features_norm = (X_features - mean_val) / std_val
    
    # Split Dataset
    indices = np.arange(len(X_features_norm))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features_norm) * 0.8)
    X_train, y_train = X_features_norm[indices[:n_train]], y_labels[indices[:n_train]]
    X_test, y_test = X_features_norm[indices[n_train:]], y_labels[indices[n_train:]]
    
    # Parameter optimal hasil eksperimen PSO sebelumnya
    optimal_C = 31.6228  # 10 ** 1.5
    optimal_gamma = 0.0316 # 10 ** -1.5
    
    logger.info(f"Melatih model SVC akhir dengan C={optimal_C:.4f}, gamma={optimal_gamma:.4f}...")
    final_model = SVC(C=optimal_C, gamma=optimal_gamma, kernel='rbf', probability=True, random_state=42)
    final_model.fit(X_train, y_train)
    
    # Evaluasi pada Test Set
    y_pred = final_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info(f"Akurasi Model pada Test Set: {acc*100:.2f}%")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    
    # Simpan Model dan Parameter Normalisasi menggunakan joblib
    model_path = os.path.join(MODEL_SAVE_DIR, "svm_seismic_model.pkl")
    scaler_path = os.path.join(MODEL_SAVE_DIR, "scaler_params.json")
    
    joblib.dump(final_model, model_path)
    
    scaler_data = {
        "mean": mean_val.tolist(),
        "std": std_val.tolist()
    }
    with open(scaler_path, 'w') as f:
        json.dump(scaler_data, f)
        
    logger.info(f" Model berhasil disimpan di: {model_path}")
    logger.info(f" Parameter Normalisasi tersimpan di: {scaler_path}")

2026-07-22 16:04:35,153 - [INFO]: === PELATIHAN DAN PENYIMPANAN MODEL FINAL PSO-SVM ===
2026-07-22 16:04:35,168 - [INFO]: Memuat file STEAD: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
Ekstraksi Fitur: 100%|██████████| 5000/5000 [00:03<00:00, 1412.33it/s]
2026-07-22 16:04:40,410 - [INFO]: Melatih model SVC akhir dengan C=31.6228, gamma=0.0316...
2026-07-22 16:04:46,429 - [INFO]: Akurasi Model pada Test Set: 72.45%
2026-07-22 16:04:46,433 - [INFO]: 
              precision    recall  f1-score   support

       Noise       0.76      0.67      0.71      1015
       Gempa       0.70      0.78      0.74       985

    accuracy                           0.72      2000
   macro avg       0.73      0.73      0.72      2000
weighted avg       0.73      0.72      0.72      2000

2026-07-22 16:04:46,446 - [INFO]:  Model berhasil disimpan di: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/saved_model_pso_svm/svm_seismic_model.pk

In [4]:
# -*- coding: utf-8 -*-
"""
Skrip Inferensi Menggunakan Model PSO-SVM yang Telah Disimpan 
pada Dataset Sinyal Seismik Indonesia
-------------------------------------------------------------------------
Skrip ini memuat model SVM dan parameter normalisasi yang telah dilatih,
kemudian melakukan inferensi (prediksi) pada data gelombang Indonesia baru.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tqdm import tqdm
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. MAIN INFERENCE PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE INFERENSI MODEL PSO-SVM PADA DATA INDONESIA ===")
    
    # Path Model yang sudah disimpan sebelumnya
    MODEL_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/saved_model_pso_svm'
    MODEL_PATH = os.path.join(MODEL_DIR, "svm_seismic_model.pkl")
    SCALER_PATH = os.path.join(MODEL_DIR, "scaler_params.json")
    
    # Path Data Inferensi Indonesia
    INFERENCE_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/output/nUUSS_embedding_1c_4_final'
    # Sesuaikan nama file JSON inferensi Bapak jika berbeda (misal: extracted_data_1c_4.json)
    # Jika direktori berisi banyak file atau satu file JSON besar, arahkan ke file JSON-nya:
    JSON_TEST_PATH = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json'
    
    NUM_POINTS = 700
    
    # Cek ketersediaan file model dan data
    if not os.path.exists(MODEL_PATH) or not os.path.exists(SCALER_PATH):
        logger.error(f"Model atau Scaler tidak ditemukan di {MODEL_DIR}. Jalankan skrip training terlebih dahulu!")
        exit()
        
    if not os.path.exists(JSON_TEST_PATH):
        logger.error(f"File data inferensi tidak ditemukan di: {JSON_TEST_PATH}")
        exit()
        
    logger.info("Memuat model PSO-SVM dan parameter normalisasi...")
    model = joblib.load(MODEL_PATH)
    
    with open(SCALER_PATH, 'r') as f:
        scaler_data = json.load(f)
    mean_val = np.array(scaler_data['mean'], dtype=np.float32)
    std_val = np.array(scaler_data['std'], dtype=np.float32)
    
    logger.info(f"Memuat data inferensi dari: {JSON_TEST_PATH}")
    with open(JSON_TEST_PATH, 'r') as f:
        test_data = json.load(f)
        
    X_features, y_true = [], []
    for key, rec in tqdm(test_data.items(), desc="Ekstraksi Fitur Data Indonesia"):
        # Evaluasi Noise (Label Aktual = 0)
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_true.append(0)
        # Evaluasi Gempa (Label Aktual = 1)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_true.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_true = np.array(y_true, dtype=np.int32)
    
    if len(X_features) == 0:
        logger.error("Tidak ada data valid yang berhasil diekstraksi untuk inferensi.")
        exit()
        
    # Normalisasi menggunakan parameter scaler dari data training
    X_features_norm = (X_features - mean_val) / std_val
    
    logger.info("Menjalankan inferensi model...")
    y_pred = model.predict(X_features_norm)
    acc = accuracy_score(y_true, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI INFERENSI MODEL PSO-SVM DI DATA INDONESIA")
    logger.info("="*50)
    logger.info(f" - Total Sampel Dievaluasi  : {len(y_true)}")
    logger.info(f" - Akurasi Pengujian (Test) : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_true, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # Simpan Visualisasi Matriks Konfusi Hasil Inferensi
    OUTPUT_VIZ_DIR = os.path.join(INFERENCE_DIR, "output_inference_pso_svm")
    os.makedirs(OUTPUT_VIZ_DIR, exist_ok=True)
    
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'PSO-SVM Inference on Indonesia Data\nAccuracy: {acc*100:.2f}%', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual (Katalog)', fontweight='bold', fontsize=12)
    
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_VIZ_DIR, "confusion_matrix_inference_indonesia.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    
    logger.info(f"Grafik matriks konfusi inferensi tersimpan di: {plot_path}")

2026-07-22 16:06:46,643 - [INFO]: === PIPELINE INFERENSI MODEL PSO-SVM PADA DATA INDONESIA ===
2026-07-22 16:06:46,644 - [INFO]: Memuat model PSO-SVM dan parameter normalisasi...
2026-07-22 16:06:46,653 - [INFO]: Memuat data inferensi dari: /Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json
Ekstraksi Fitur Data Indonesia: 100%|██████████| 5601/5601 [00:04<00:00, 1396.66it/s]
2026-07-22 16:06:52,531 - [INFO]: Menjalankan inferensi model...
2026-07-22 16:06:54,620 - [INFO]: 
2026-07-22 16:06:54,620 - [INFO]: HASIL EVALUASI INFERENSI MODEL PSO-SVM DI DATA INDONESIA
2026-07-22 16:06:54,620 - [INFO]: ==================================================
2026-07-22 16:06:54,620 - [INFO]:  - Total Sampel Dievaluasi  : 11202
2026-07-22 16:06:54,621 - [INFO]:  - Akurasi Pengujian (Test) : 55.59%
2026-07-22 16:06:54,621 - [INFO]: 
Classification Report:
2026-07-22 16:06:54,626 - [INFO]: 
              precision    recall  f1-score   support

       Noise       0.53      

In [5]:
# -*- coding: utf-8 -*-
"""
Optimasi Hyperparameter SVM Tingkat Lanjut Menggunakan Particle Swarm Optimization (PSO)
beserta Serialisasi (Saving) Model Optimal untuk Inferensi
-------------------------------------------------------------------------
Skrip ini memperluas ruang pencarian C dan gamma, melatih SVM dengan PSO, 
kemudian menyimpan model terbaik dan parameter normalisasi ke dalam disk.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. IMPLEMENTASI PSO LANJUTAN UNTUK SVM
# ==============================================================================
class AdvancedParticle:
    def __init__(self, bounds):
        self.dim = len(bounds)
        self.position = np.array([random.uniform(b[0], b[1]) for b in bounds])
        self.velocity = np.array([random.uniform(-(b[1]-b[0])*0.15, (b[1]-b[0])*0.15) for b in bounds])
        self.best_position = np.copy(self.position)
        self.best_fitness = -float('inf')
        self.fitness = -float('inf')
        self.bounds = bounds

    def update_position(self):
        for i in range(self.dim):
            self.position[i] += self.velocity[i]
            self.position[i] = max(self.bounds[i][0], min(self.position[i], self.bounds[i][1]))

def pso_optimize_svm_advanced(X_train, y_train, X_val, y_val, num_particles=20, max_iter=25):
    logger.info("Memulai optimasi lanjutan SVM menggunakan PSO (Ruang parameter diperluas)...")
    
    # Memperluas rentang log10: C [0.001, 1000] -> [-3.0, 3.0], gamma [0.0001, 100] -> [-4.0, 2.0]
    bounds = [(-3.0, 3.0), (-4.0, 2.0)]
    
    particles = [AdvancedParticle(bounds) for _ in range(num_particles)]
    global_best_position = np.zeros(2)
    global_best_fitness = -float('inf')
    
    # Adaptive Inertia Weight
    w_max, w_min = 0.9, 0.4
    
    for iteration in range(max_iter):
        w = w_max - (w_max - w_min) * (iteration / max_iter)
        
        for particle in particles:
            C_val = 10 ** particle.position[0]
            gamma_val = 10 ** particle.position[1]
            
            try:
                svm = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
                svm.fit(X_train, y_train)
                preds = svm.predict(X_val)
                fitness = accuracy_score(y_val, preds)
            except Exception:
                fitness = 0.0
                
            particle.fitness = fitness
            if particle.fitness > particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = np.copy(particle.position)
                
            if particle.fitness > global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = np.copy(particle.position)
                
        c1, c2 = 1.5, 1.5
        for particle in particles:
            for i in range(len(bounds)):
                r1, r2 = random.random(), random.random()
                cognitive = c1 * r1 * (particle.best_position[i] - particle.position[i])
                social = c2 * r2 * (global_best_position[i] - particle.position[i])
                particle.velocity[i] = w * particle.velocity[i] + cognitive + social
            particle.update_position()
            
        logger.info(f"Iterasi PSO Advanced {iteration+1}/{max_iter} | Best Val Accuracy: {global_best_fitness*100:.2f}%")
        
    return 10 ** global_best_position[0], 10 ** global_best_position[1]

# ==============================================================================
# 3. MAIN PIPELINE DENGAN PENYIMPANAN MODEL (SAVING)
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE PSO-SVM LANJUTAN + SERIALISASI MODEL ===")
    
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_20260719_062844.json'
    JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    NUM_POINTS = 700
    
    SAVE_DIR = os.path.join(KEY_DATA_DIR, "output_pso_svm_optimized")
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    if not os.path.exists(JSON_PATH):
        logger.error(f"File JSON STEAD tidak ditemukan di: {JSON_PATH}")
        exit()
        
    logger.info(f"Memuat file STEAD: {JSON_PATH}")
    with open(JSON_PATH, 'r') as f:
        stead_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(stead_data.items(), desc="Ekstraksi Fitur STEAD"):
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Hitung dan simpan parameter normalisasi (mean & std)
    mean_val = np.mean(X_features, axis=0)
    std_val = np.std(X_features, axis=0) + 1e-9
    X_features_norm = (X_features - mean_val) / std_val
    
    # Split Dataset: Train (60%), Validation (20%), Test (20%)
    indices = np.arange(len(X_features_norm))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_train = int(len(X_features_norm) * 0.6)
    n_val = int(len(X_features_norm) * 0.2)
    
    X_train, y_train = X_features_norm[indices[:n_train]], y_labels[indices[:n_train]]
    X_val, y_val = X_features_norm[indices[n_train:n_train+n_val]], y_labels[indices[n_train:n_train+n_val]]
    X_test, y_test = X_features_norm[indices[n_train+n_val:]], y_labels[indices[n_train+n_val:]]
    
    # Jalankan PSO Lanjutan
    best_C, best_gamma = pso_optimize_svm_advanced(X_train, y_train, X_val, y_val, num_particles=20, max_iter=25)
    logger.info(f"Parameter Optimal Baru -> C: {best_C:.4f}, gamma: {best_gamma:.4f}")
    
    # Pelatihan Akhir (Train + Val) & Evaluasi
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.hstack([y_train, y_val])
    
    final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', probability=True, random_state=42)
    final_svm.fit(X_train_full, y_train_full)
    
    y_pred = final_svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI MODEL PSO-SVM OPTIMIZED")
    logger.info("="*50)
    logger.info(f" - Akurasi Pengujian Test Set : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # ==============================================================================
    # 4. SERIALISASI (SAVING) MODEL DAN SCALER KE DISK
    # ==============================================================================
    model_filename = "svm_seismic_model_optimized.pkl"
    scaler_filename = "scaler_params_optimized.json"
    
    model_path = os.path.join(SAVE_DIR, model_filename)
    scaler_path = os.path.join(SAVE_DIR, scaler_filename)
    
    joblib.dump(final_svm, model_path)
    
    scaler_data = {
        "mean": mean_val.tolist(),
        "std": std_val.tolist(),
        "optimal_C": float(best_C),
        "optimal_gamma": float(best_gamma),
        "test_accuracy": float(acc)
    }
    with open(scaler_path, 'w') as f:
        json.dump(scaler_data, f, indent=4)
        
    logger.info(f" Model SVM optimal berhasil disimpan di : {model_path}")
    logger.info(f" Parameter normalisasi tersimpan di    : {scaler_path}")

    # Visualisasi
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'Optimized PSO-SVM on STEAD\nAccuracy: {acc*100:.2f}% | C={best_C:.2f}, gamma={best_gamma:.2f}', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual', fontweight='bold', fontsize=12)
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, "confusion_matrix_pso_svm_optimized.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    logger.info(f"Grafik visualisasi tersimpan di: {plot_path}")

2026-07-22 16:08:02,267 - [INFO]: === PIPELINE PSO-SVM LANJUTAN + SERIALISASI MODEL ===
2026-07-22 16:08:02,268 - [INFO]: Memuat file STEAD: /Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json
Ekstraksi Fitur STEAD: 100%|██████████| 5000/5000 [00:03<00:00, 1307.70it/s]
2026-07-22 16:08:07,905 - [INFO]: Memulai optimasi lanjutan SVM menggunakan PSO (Ruang parameter diperluas)...
2026-07-22 16:08:30,910 - [INFO]: Iterasi PSO Advanced 1/25 | Best Val Accuracy: 78.75%
2026-07-22 16:08:55,126 - [INFO]: Iterasi PSO Advanced 2/25 | Best Val Accuracy: 80.40%
2026-07-22 16:09:28,279 - [INFO]: Iterasi PSO Advanced 3/25 | Best Val Accuracy: 80.40%
2026-07-22 16:10:10,178 - [INFO]: Iterasi PSO Advanced 4/25 | Best Val Accuracy: 80.40%
2026-07-22 16:11:05,262 - [INFO]: Iterasi PSO Advanced 5/25 | Best Val Accuracy: 80.75%
2026-07-22 16:11:47,828 - [INFO]: Iterasi PSO Advanced 6/25 | Best Val Accuracy: 80.75%
2026-07-22 16:12:29,406 - [INFO]: Iterasi PSO

In [6]:
# -*- coding: utf-8 -*-
"""
Skrip Inferensi Menggunakan Model PSO-SVM Teroptimasi 
pada Dataset Sinyal Seismik Indonesia (1C)
-------------------------------------------------------------------------
Skrip ini memuat model SVM optimal dan parameter normalisasi yang telah disimpan,
kemudian melakukan inferensi (prediksi) pada data gelombang Indonesia baru.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tqdm import tqdm
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. MAIN INFERENCE PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE INFERENSI MODEL PSO-SVM OPTIMIZED PADA DATA INDONESIA ===")
    
    # Path Model dan Scaler Teroptimasi
    MODEL_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/output_pso_svm_optimized'
    MODEL_PATH = os.path.join(MODEL_DIR, "svm_seismic_model_optimized.pkl")
    SCALER_PATH = os.path.join(MODEL_DIR, "scaler_params_optimized.json")
    
    # Path Data Inferensi Indonesia yang Bapak berikan
    KEY_DATA_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4'
    KEY_TEST_FILE = "extracted_data_1c_4_final.json"
    JSON_TEST_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    
    NUM_POINTS = 700
    
    # Direktori Output Hasil Inferensi
    OUTPUT_VIZ_DIR = os.path.join(KEY_DATA_DIR, "output_inference_pso_svm_optimized")
    os.makedirs(OUTPUT_VIZ_DIR, exist_ok=True)
    
    # Cek ketersediaan file model dan data
    if not os.path.exists(MODEL_PATH) or not os.path.exists(SCALER_PATH):
        logger.error(f"Model atau Scaler teroptimasi tidak ditemukan di: {MODEL_DIR}")
        exit()
        
    if not os.path.exists(JSON_TEST_PATH):
        logger.error(f"File data inferensi Indonesia tidak ditemukan di: {JSON_TEST_PATH}")
        exit()
        
    logger.info("Memuat model PSO-SVM optimized dan parameter normalisasi...")
    model = joblib.load(MODEL_PATH)
    
    with open(SCALER_PATH, 'r') as f:
        scaler_data = json.load(f)
    mean_val = np.array(scaler_data['mean'], dtype=np.float32)
    std_val = np.array(scaler_data['std'], dtype=np.float32)
    
    logger.info(f"Memuat data inferensi dari: {JSON_TEST_PATH}")
    with open(JSON_TEST_PATH, 'r') as f:
        test_data = json.load(f)
        
    X_features, y_true = [], []
    for key, rec in tqdm(test_data.items(), desc="Ekstraksi Fitur Data Indonesia"):
        # Evaluasi Noise (Label Aktual = 0)
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_true.append(0)
        # Evaluasi Gempa (Label Aktual = 1)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_true.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_true = np.array(y_true, dtype=np.int32)
    
    if len(X_features) == 0:
        logger.error("Tidak ada data valid yang berhasil diekstraksi untuk inferensi.")
        exit()
        
    # Normalisasi menggunakan parameter scaler dari data training STEAD
    X_features_norm = (X_features - mean_val) / std_val
    
    logger.info("Menjalankan inferensi model pada dataset Indonesia...")
    y_pred = model.predict(X_features_norm)
    acc = accuracy_score(y_true, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI INFERENSI PSO-SVM OPTIMIZED DI DATA INDONESIA")
    logger.info("="*50)
    logger.info(f" - Total Sampel Dievaluasi  : {len(y_true)}")
    logger.info(f" - Akurasi Pengujian (Test) : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_true, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # Simpan Visualisasi Matriks Konfusi Hasil Inferensi
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'Optimized PSO-SVM Inference on Indonesia Data\nAccuracy: {acc*100:.2f}%', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual (Katalog)', fontweight='bold', fontsize=12)
    
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_VIZ_DIR, "confusion_matrix_inference_indonesia_optimized.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    
    logger.info(f"Grafik matriks konfusi inferensi tersimpan di: {plot_path}")

2026-07-22 18:20:12,611 - [INFO]: === PIPELINE INFERENSI MODEL PSO-SVM OPTIMIZED PADA DATA INDONESIA ===
2026-07-22 18:20:12,653 - [INFO]: Memuat model PSO-SVM optimized dan parameter normalisasi...
2026-07-22 18:20:12,662 - [INFO]: Memuat data inferensi dari: /Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json
Ekstraksi Fitur Data Indonesia: 100%|██████████| 5601/5601 [00:03<00:00, 1412.01it/s]
2026-07-22 18:20:18,590 - [INFO]: Menjalankan inferensi model pada dataset Indonesia...
2026-07-22 18:20:19,955 - [INFO]: 
2026-07-22 18:20:19,955 - [INFO]: HASIL EVALUASI INFERENSI PSO-SVM OPTIMIZED DI DATA INDONESIA
2026-07-22 18:20:19,956 - [INFO]: ==================================================
2026-07-22 18:20:19,956 - [INFO]:  - Total Sampel Dievaluasi  : 11202
2026-07-22 18:20:19,956 - [INFO]:  - Akurasi Pengujian (Test) : 52.65%
2026-07-22 18:20:19,956 - [INFO]: 
Classification Report:
2026-07-22 18:20:19,962 - [INFO]: 
              precision    recall  f1

In [7]:
# -*- coding: utf-8 -*-
"""
Fine-Tuning Model PSO-SVM STEAD pada Data Seismik Indonesia
-------------------------------------------------------------------------
Skrip ini memuat model PSO-SVM yang telah dilatih pada dataset STEAD,
kemudian melakukan fine-tuning (adaptasi domain) menggunakan sampel data lokal Indonesia
untuk mengatasi penurunan performa akibat domain shift.
"""

import os
import numpy as np
import json
import logging
from scipy.stats import kurtosis, skew
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tqdm import tqdm
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

# ==============================================================================
# 1. EKSTRAKSI FITUR STATISTIK SEISMIK
# ==============================================================================
def extract_statistical_features(signal):
    sig = np.array(signal, dtype=np.float32)
    if len(sig) == 0:
        return np.zeros(5)
    
    rms = np.sqrt(np.mean(sig**2))
    kurt = kurtosis(sig)
    sk = skew(sig)
    zcr = np.mean(np.diff(np.signbit(sig)))
    max_val = np.max(np.abs(sig))
    par = max_val / (rms + 1e-9)
    
    return np.array([rms, kurt, sk, zcr, par])

# ==============================================================================
# 2. MAIN FINE-TUNING PIPELINE
# ==============================================================================
if __name__ == "__main__":
    logger.info("=== PIPELINE FINE-TUNING MODEL PSO-SVM UNTUK INDONESIA ===")
    
    # Path Model STEAD Terdahulu
    STEAD_MODEL_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/output_pso_svm_optimized'
    MODEL_PATH = os.path.join(STEAD_MODEL_DIR, "svm_seismic_model_optimized.pkl")
    SCALER_PATH = os.path.join(STEAD_MODEL_DIR, "scaler_params_optimized.json")
    
    # Path Data Indonesia
    KEY_DATA_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4'
    KEY_TEST_FILE = "extracted_data_1c_4_final.json"
    JSON_TEST_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)
    
    NUM_POINTS = 700
    SAVE_DIR = os.path.join(KEY_DATA_DIR, "output_finetuned_model_indonesia")
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    if not os.path.exists(MODEL_PATH) or not os.path.exists(JSON_TEST_PATH):
        logger.error("File model STEAD atau file data Indonesia tidak ditemukan!")
        exit()
        
    logger.info("Memuat model dasar PSO-SVM dari STEAD...")
    base_model = joblib.load(MODEL_PATH)
    
    with open(SCALER_PATH, 'r') as f:
        stead_scaler = json.load(f)
    stead_mean = np.array(stead_scaler['mean'], dtype=np.float32)
    stead_std = np.array(stead_scaler['std'], dtype=np.float32)
    
    logger.info(f"Memuat data Indonesia dari: {JSON_TEST_PATH}")
    with open(JSON_TEST_PATH, 'r') as f:
        indo_data = json.load(f)
        
    X_features, y_labels = [], []
    for key, rec in tqdm(indo_data.items(), desc="Ekstraksi Fitur Indonesia"):
        if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
            zn = rec["Z_noise"][-NUM_POINTS:]
            X_features.append(extract_statistical_features(zn))
            y_labels.append(0)
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            zs = rec["Z"][:NUM_POINTS]
            X_features.append(extract_statistical_features(zs))
            y_labels.append(1)
            
    X_features = np.array(X_features, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    # Hitung mean & std baru berdasarkan karakteristik lokal Indonesia untuk normalisasi domain
    indo_mean = np.mean(X_features, axis=0)
    indo_std = np.std(X_features, axis=0) + 1e-9
    
    # Normalisasi fitur menggunakan parameter lokal Indonesia (Domain Adaptation)
    X_features_norm = (X_features - indo_mean) / indo_std
    
    # Split Data Indonesia untuk Fine-Tuning (20% untuk adaptasi/fine-tuning, 80% untuk evaluasi murni)
    indices = np.arange(len(X_features_norm))
    np.random.seed(42)
    np.random.shuffle(indices)
    
    n_finetune = int(len(X_features_norm) * 0.2)
    
    ft_idx = indices[:n_finetune]
    test_idx = indices[n_finetune:]
    
    X_ft, y_ft = X_features_norm[ft_idx], y_labels[ft_idx]
    X_test, y_test = X_features_norm[test_idx], y_labels[test_idx]
    
    logger.info(f"Jumlah sampel Fine-Tuning (Adaptasi): {len(X_ft)}")
    logger.info(f"Jumlah sampel Pengujian (Test): {len(X_test)}")
    
    # Proses Fine-Tuning menggunakan warm_start=True pada SVM
    logger.info("Melakukan Fine-Tuning model SVM dengan subset data lokal Indonesia...")
    base_model.warm_start = True
    base_model.fit(X_ft, y_ft)  # Melatih ulang incremental pada domain lokal
    
    # Evaluasi pada Sisa Test Set Indonesia
    y_pred = base_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI FINE-TUNING MODEL PADA DATA INDONESIA")
    logger.info("="*50)
    logger.info(f" - Akurasi Test Set Indonesia : {acc*100:.2f}%")
    logger.info("\nClassification Report:")
    logger.info("\n" + classification_report(y_test, y_pred, target_names=['Noise', 'Gempa']))
    logger.info("="*50)

    # Simpan Model dan Scaler Hasil Fine-Tuning
    model_path = os.path.join(SAVE_DIR, "svm_seismic_model_finetuned.pkl")
    scaler_path = os.path.join(SAVE_DIR, "scaler_params_finetuned.json")
    
    joblib.dump(base_model, model_path)
    scaler_data = {
        "mean": indo_mean.tolist(),
        "std": indo_std.tolist(),
        "test_accuracy": float(acc)
    }
    with open(scaler_path, 'w') as f:
        json.dump(scaler_data, f, indent=4)
        
    logger.info(f" Model Fine-Tuned tersimpan di : {model_path}")
    logger.info(f" Parameter Scaler Lokal tersimpan di: {scaler_path}")

    # Visualisasi
    plt.figure(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', annot_kws={"size": 16, "weight": "bold"})
    plt.title(f'Fine-Tuned PSO-SVM on Indonesia Data\nAccuracy: {acc*100:.2f}%', fontweight='bold', fontsize=12)
    plt.xlabel('Prediksi Model', fontweight='bold', fontsize=12)
    plt.ylabel('Aktual', fontweight='bold', fontsize=12)
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(['Noise', 'Gempa'])
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(['Noise', 'Gempa'])
    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, "confusion_matrix_finetuned_indonesia.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    logger.info(f"Grafik visualisasi tersimpan di: {plot_path}")

2026-07-22 18:23:19,367 - [INFO]: === PIPELINE FINE-TUNING MODEL PSO-SVM UNTUK INDONESIA ===
2026-07-22 18:23:19,389 - [INFO]: Memuat model dasar PSO-SVM dari STEAD...
2026-07-22 18:23:19,397 - [INFO]: Memuat data Indonesia dari: /Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json
Ekstraksi Fitur Indonesia: 100%|██████████| 5601/5601 [00:04<00:00, 1398.52it/s]
2026-07-22 18:23:25,269 - [INFO]: Jumlah sampel Fine-Tuning (Adaptasi): 2240
2026-07-22 18:23:25,270 - [INFO]: Jumlah sampel Pengujian (Test): 8962
2026-07-22 18:23:25,270 - [INFO]: Melakukan Fine-Tuning model SVM dengan subset data lokal Indonesia...
2026-07-22 18:23:28,005 - [INFO]: 
2026-07-22 18:23:28,005 - [INFO]: HASIL EVALUASI FINE-TUNING MODEL PADA DATA INDONESIA
2026-07-22 18:23:28,005 - [INFO]: ==================================================
2026-07-22 18:23:28,006 - [INFO]:  - Akurasi Test Set Indonesia : 58.66%
2026-07-22 18:23:28,006 - [INFO]: 
Classification Report:
2026-07-22 18:23:28,